In [62]:
from dotenv import load_dotenv , find_dotenv
import os 
load_dotenv(find_dotenv(), override=True)

True

## Basic Prompt
it's not flexible because it limited to length of model

In [4]:
from langchain.chat_models import ChatOpenAI
from langchain.schema import (
AIMessage, 
HumanMessage,
SystemMessage
)

In [5]:
text= """
Mojo combines the usability of Python with the performance of C, unlocking unparalleled programmability \
of AI hardware and extensibility of AI models.
Mojo is a new programming language that bridges the gap between research and production \ 
by combining the best of Python syntax with systems programming and metaprogramming.
With Mojo, you can write portable code that’s faster than C and seamlessly inter-op with the Python ecosystem.
When we started Modular, we had no intention of building a new programming language. \
But as we were building our platform with the intent to unify the world’s ML/AI infrastructure, \
we realized that programming across the entire stack was too complicated. Plus, we were writing a \
lot of MLIR by hand and not having a good time.
And although accelerators are important, one of the most prevalent and sometimes overlooked "accelerators" \
is the host CPU. Nowadays, CPUs have lots of tensor-core-like accelerator blocks and other AI acceleration \
units, but they also serve as the “fallback” for operations that specialized accelerators don’t handle, \
such as data loading, pre- and post-processing, and integrations with foreign systems. \
"""

In [6]:
messages = [
    SystemMessage(content='You are an Expert Copywriter with expertize in summerizing documents'),
    HumanMessage(content=f'Please Provide a short and concise summary of the following Text: \n Text : {text}'),
]

llm = ChatOpenAI(model_name='gpt-3.5-turbo', temperature=0)


In [7]:
llm.get_num_tokens(text)

231

In [8]:
summary_output = llm(messages)

In [9]:
print(summary_output.content)

Mojo is a new programming language that combines the usability of Python with the performance of C. It aims to bridge the gap between research and production in the field of AI by offering portable code that is faster than C and seamlessly integrates with the Python ecosystem. Mojo was developed by Modular to simplify programming across the entire ML/AI infrastructure and address the complexity of writing MLIR by hand. It recognizes the importance of host CPUs as accelerators and provides support for operations that specialized accelerators may not handle, such as data loading, pre- and post-processing, and integrations with foreign systems.


## Prompt Template

In [10]:
# it only summerizes text that has lower length

from langchain import PromptTemplate
from langchain.chains import LLMChain


In [11]:
template = """
write a concise and short summary of the following text:
Text : {text}
translate the summary to {language}
"""

prompt = PromptTemplate(input_variables=['text', 'language'],
                       template=template)

In [12]:
llm.get_num_tokens(prompt.format(text=text, language='arabic'))

252

In [13]:
chain = LLMChain(llm=llm, prompt=prompt)

In [14]:
result = chain.run({"text":text, "language":"arabic"})
print(result)

موجو هي لغة برمجة جديدة تجمع بين سهولة استخدام Python وأداء C، مما يتيح إمكانية برمجة غير مسبوقة لأجهزة الذكاء الاصطناعي وقابلية توسيع نماذج الذكاء الاصطناعي. تعمل موجو على تقديم رمز قابل للنقل أسرع من C وتكامل سلس مع بيئة Python. تهدف موجو إلى توفير جسر بين البحث والإنتاجية من خلال دمج أفضل بناء جملة في Python مع برمجة الأنظمة والبرمجة الذاتية.


## Stuff Document

In [18]:
from langchain import PromptTemplate
from langchain.chat_models import ChatOpenAI
from langchain.chains.summarize import load_summarize_chain
from langchain.docstore.document import Document

In [30]:
with open("sj.txt", encoding='utf-8') as f:
    text = f.read()

docs = [Document(page_content=text)]

llm = ChatOpenAI(model_name='gpt-3.5-turbo', temperature=0)

In [31]:
template = """ Write a concise and short summary of the following text
Text: `{text}` 
"""
prompt = PromptTemplate(input_variables=['text'],
                       template=template)

In [32]:
chain = load_summarize_chain(
    llm,
    chain_type='stuff',
    prompt=prompt,
    verbose=False
)

output_summary = chain.run(docs)

In [33]:
print(output_summary)

The speaker, who never graduated from college, shares three stories from his life. The first story is about dropping out of college and how it led him to take a calligraphy class, which later influenced the design of the Macintosh computer. The second story is about getting fired from the company he co-founded, Apple, and how it led him to start new ventures and find love. The third story is about facing death when he was diagnosed with cancer and how it changed his perspective on life. The speaker encourages the audience to follow their hearts, not waste time, and stay hungry and foolish.


# **Summarizing Large Documents using Map_reuce** 

In [76]:
from langchain import PromptTemplate
from langchain.chat_models import ChatOpenAI
from langchain.chains.summarize import load_summarize_chain
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [77]:
with open("sj.txt", encoding='utf-8') as f:
    text = f.read()

llm = ChatOpenAI(model_name='gpt-3.5-turbo', temperature=0)

In [78]:
llm.get_num_tokens(text)

2653

In [79]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=10000, chunk_overlap=50)
chunks = text_splitter.create_documents([text])

In [80]:
len(chunks)

2

In [83]:
chain = load_summarize_chain(llm,
                            chain_type='map_reduce',
                            verbose=False)
output_summary = chain.run(chunks)

In [84]:
output_summary

"Steve Jobs shares three stories from his life, including dropping out of college and how it influenced the design of the Macintosh computer, getting fired from Apple and finding success in new ventures, and his experience with cancer and the importance of living each day fully. He encourages the audience to follow their passions, not settle, and remember that life is short. The speaker also reflects on the inevitability of death, urges the audience to live their own lives, and mentions The Whole Earth Catalog's message of staying hungry and foolish."

In [85]:
chain.llm_chain.prompt.template

'Write a concise summary of the following:\n\n\n"{text}"\n\n\nCONCISE SUMMARY:'

In [86]:
chain.combine_document_chain.llm_chain.prompt.template

'Write a concise summary of the following:\n\n\n"{text}"\n\n\nCONCISE SUMMARY:'

## **Map Reduce with custom Prompt**

In [89]:
map_prompt = """ Write a short and conise summary of the following:
Text:`{text}`
Concise Summary:
"""

map_prompt_template = PromptTemplate(input_variables=['text'],
                                     template=map_prompt)

In [90]:
combine_prompt = '''
Write a concise summary of the following text that covers the key points.
Add a title to the summary.
Start your summary with an INTRODUCTION PARAGRAPH that gives an overview of the topic FOLLOWED
by BULLET POINTS if possible AND end the summary with a CONCLUSION PHRASE.
Text: `{text}`
'''
combine_prompt_template = PromptTemplate(template=combine_prompt,
                                        input_variables=['text'])

In [93]:
summary_chain = load_summarize_chain(llm=llm,
                                    chain_type='map_reduce',
                                    map_prompt=map_prompt_template,
                                    combine_prompt=combine_prompt_template,
                                    verbose=False)
output = summary_chain.run(chunks)
print(output)

Title: Lessons from Life: Connecting the Dots, Love and Loss, and Facing Death

Introduction:
The speaker in a commencement speech shares three impactful stories from their life, highlighting the importance of connecting the dots, embracing love and loss, and facing death.

Key Points:
- Connecting the dots: Trusting that things will work out in the future by connecting the experiences and opportunities in life.
- Love and loss: Getting fired from a company leads to new opportunities and personal growth, emphasizing the importance of embracing change.
- Facing death: The speaker's personal experience with facing death influences their perspective on life, emphasizing the need to live one's own life and not be influenced by others' opinions.
- The inevitability of death: Discussing the certainty of death and encouraging readers to make the most of their lives.
- Following intuition: Encouraging readers to trust their own intuition and not be swayed by others' opinions.
- Personal anecdo